# Expense Tracker — Kaggle Master Training

This notebook installs the `ml/` package, shows attached Kaggle inputs, configures CPU/GPU resources, and runs the master training pipeline.

Attach as many Kaggle datasets as needed through **Add Input**. The files are exposed under `/kaggle/input/<dataset>/...`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/kaggle/working/expense-tracker')
BRANCH = 'feature/ml-expense-intelligence'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yoge-2004/expense-tracker.git', str(REPO)], check=True)
ML = REPO / 'ml'
sys.path.insert(0, str(ML / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ML)], check=True)
print('ML root:', ML)

In [ ]:
from expense_ml.resources import configure_resources

resources = configure_resources()
print(resources)

for path in sorted(Path('/kaggle/input').glob('*')):
    print(path)

## Optional: use multiple Kaggle dataset files

Set `KAGGLE_DATASETS` for attached CSV/Parquet files. Each item needs its source name and the text/label column names. Keep the paths under `/kaggle/input`.

In [ ]:
import yaml

KAGGLE_DATASETS = [
    # {'source': 'global', 'path': '/kaggle/input/global-transaction-categorization/data.csv',
    #  'text_column': 'transaction_description', 'label_column': 'category'},
    # {'source': 'finee-india', 'path': '/kaggle/input/finee-dataset/train.csv',
    #  'text_column': 'input', 'label_column': 'output.category'},
]

config = yaml.safe_load((ML / 'config' / 'datasets.yaml').read_text())
if KAGGLE_DATASETS:
    config['datasets'] = KAGGLE_DATASETS
    kaggle_config = Path('/kaggle/working/kaggle_datasets.yaml')
    kaggle_config.write_text(yaml.safe_dump(config, sort_keys=False))
    os.environ['EXPENSE_ML_CONFIG'] = str(kaggle_config)
    print(kaggle_config.read_text())
else:
    os.environ['EXPENSE_ML_CONFIG'] = str(ML / 'config' / 'datasets.yaml')
    print('Using repository dataset config.')

In [ ]:
# Adjust these only when the attached GPU/RAM requires it.
os.environ.setdefault('EXPENSE_ML_CPU_THREADS', 'auto')
os.environ.setdefault('EXPENSE_ML_DATALOADER_WORKERS', '8')
os.environ.setdefault('EXPENSE_ML_BATCH_SIZE', '32')
os.environ.setdefault('EXPENSE_ML_EVAL_BATCH_SIZE', '64')
os.environ.setdefault('EXPENSE_ML_MIXED_PRECISION', 'auto')
os.environ.setdefault('EXPENSE_ML_MAX_MERCHANTS', '250000')
os.environ.setdefault('EXPENSE_ML_DUPLICATE_MAX_ROWS', '500000')
os.environ.setdefault('EXPENSE_ML_NORMALIZE_CHUNK_SIZE', '250000')
os.environ['EXPENSE_ML_OUTPUT'] = '/kaggle/working/expense-ml-runs'
print({k:v for k,v in os.environ.items() if k.startswith('EXPENSE_ML_')})

In [ ]:
# Full run: TF-IDF + Transformer + auxiliary models + reports/figures.
%run /kaggle/working/expense-tracker/ml/kaggle_train.py

In [ ]:
# Inspect the latest run.
from pathlib import Path
import json

runs = sorted(Path('/kaggle/working/expense-ml-runs').glob('*'))
latest = runs[-1]
print('Latest run:', latest)
print((latest / 'reports' / 'REPORT.md').read_text())
print(json.loads((latest / 'manifest.json').read_text())['resources'])